# CPSA 08 — build the IR-look full-frame hand dataset and train the top-down hand model

The deployed hand model sees the whole 640x480 INZI IR frame, not a crop: it has
to find every hand itself and return 21 MediaPipe landmarks.  This notebook
builds that dataset from the same two sources as notebook 07 and trains the
model on it.

Sources (both downloaded here, no Kaggle dataset attached):

* **HaGRIDv2** — official `annotations_with_landmarks` JSON (21 MediaPipe
  landmarks per hand) plus the 30k-image 384p sample
  (`cj-mills/hagrid-sample-30k-384p`).
  License: CC BY-SA 4.0 variant (share-alike, attribution).
* **Ultralytics hand-keypoints** — 26,768 images with 21-keypoint YOLO labels.
  License: **CC BY-NC-SA 4.0 — non-commercial.**  Set `USE_ULTRALYTICS = False`
  to build a HaGRID-only dataset that is not encumbered by the NC clause, and
  do not publish a dataset that mixes the two without clearing that license.

The geometry and the look live in `cpsa_ml.datasets.handframe`: one window per
source image (5:4 aspect, sized so the largest hand spans 25-60% of the window
width, grown until it covers every hand), resized to 256x320, converted to
grayscale and replicated to three equal channels, then the IR look in order —
gamma 0.6-1.6, contrast 0.6-1.4, Gaussian noise sigma 0-8, Gaussian blur with
probability 0.3, and the runtime CLAHE (clipLimit 3.0, tiles 8x8) with
probability 0.8.  Every hand inside the window is labelled; an image whose
annotation holds a hand that cannot be labelled is skipped, so the model never
learns to ignore a hand.  One class `hand`, 21 keypoints, identity `flip_idx`.

Training reuses `cpsa_ml.train` with the DPU-friendly LeakyReLU(26/256) config
`configs/yolov8n-hand-lrelu.yaml`, `yolov8n`, square `imgsz=320`, batch 64 on
2x T4, `hsv_h=0`/`hsv_s=0` (the frames are gray) and `degrees=180` (a hand seen
from above has no canonical orientation).  The exported graph is 256x320, the
input of the compiled model.

Set `COMMIT_SHA` below to the exact repository commit for this run.  Outputs:
`/kaggle/working/hand-topdown` (dataset, re-publish as
`fabiocabeccia/cpsa-hand-topdown`) and `/kaggle/working/hand-topdown-model`
(weights, ONNX at 256x320 and the dataset metadata).

## Outline

1. Pin and install `cpsa-ml` at `COMMIT_SHA`.
2. Download and extract HaGRID + hand-keypoints.
3. Build the dataset with `build_handframe_dataset`.
4. Train with `TrainConfig` + `train`.
5. Export `best.pt`, the 256x320 ONNX and the dataset metadata.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "b331c1ee887f544c9915be1e405b2376cede33d1"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"cpsa-ml @ git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
import subprocess
from pathlib import Path

DATA = Path("/tmp/hand_data")  # outside /kaggle/working: not part of the output
DATA.mkdir(parents=True, exist_ok=True)

URLS = {
    "hagrid-384p.zip": (
        "https://huggingface.co/datasets/cj-mills/hagrid-sample-30k-384p"
        "/resolve/main/hagrid-sample-30k-384p.zip"
    ),
    "annotations.zip": (
        "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets"
        "/hagrid_v2/annotations_with_landmarks/annotations.zip"
    ),
    "hand-keypoints.zip": (
        "https://github.com/ultralytics/assets/releases/download/v0.0.0/hand-keypoints.zip"
    ),
}

# ~2.1 GB in total; curl -C - resumes a download if the session dies.
for name, url in URLS.items():
    archive = DATA / name
    if not archive.is_file():
        subprocess.run(
            ["curl", "-sSL", "--retry", "3", "-C", "-", "-o", str(archive), url], check=True
        )
    print(name, round(archive.stat().st_size / 1e6, 1), "MB")

In [ ]:
import zipfile
from pathlib import Path

# The 384p sample is one top-level folder with one sub-directory per gesture.
with zipfile.ZipFile(DATA / "hagrid-384p.zip") as archive:
    archive.extractall(DATA / "hagrid")
HAGRID_ROOT = DATA / "hagrid" / "hagrid-sample-30k-384p" / "hagrid_30k"
GESTURES = sorted(p.name.split("train_val_")[-1] for p in HAGRID_ROOT.iterdir() if p.is_dir())

# Only the annotation files of the gestures present in the sample: the whole
# tree is ~1.5 GB and most of it is never read.
annotations_zip = zipfile.ZipFile(DATA / "annotations.zip")
members = [
    name
    for name in annotations_zip.namelist()
    if name.startswith("annotations/") and name.endswith(".json") and Path(name).stem in GESTURES
]
annotations_zip.extractall(DATA / "hagrid_annotations", members=members)
ANNOTATIONS = DATA / "hagrid_annotations" / "annotations"

ULTRALYTICS_ROOT = DATA / "hand-keypoints"
if not (ULTRALYTICS_ROOT / "labels" / "train").is_dir():
    with zipfile.ZipFile(DATA / "hand-keypoints.zip") as archive:
        archive.extractall(ULTRALYTICS_ROOT)

print("gestures:", GESTURES)
print("images:", sum(1 for _ in HAGRID_ROOT.rglob("*.jpg")))
print("annotation files:", sorted(p.name for p in ANNOTATIONS.rglob("*.json"))[:5], "...")
print(
    "hand-keypoints train images:",
    sum(1 for _ in (ULTRALYTICS_ROOT / "images" / "train").glob("*.jpg")),
)

In [ ]:
import json
from pathlib import Path

from cpsa_ml.datasets.handframe import HandFrameConfig, build_handframe_dataset
from cpsa_ml.datasets.hands import HagridSource, UltralyticsHandSource

OUT = Path("/kaggle/working/hand-topdown")
USE_ULTRALYTICS = True  # False -> HaGRID-only dataset, no CC BY-NC-SA source
VAL_RATIO = 0.05  # share of the source train images held out for validation
SEED = 0

stats = build_handframe_dataset(
    HandFrameConfig(
        out=OUT,
        imgsz_hw=(256, 320),
        hand_frac=(0.25, 0.6),
        clahe_p=0.8,
        seed=SEED,
        val_ratio=VAL_RATIO,
    ),
    hagrid=HagridSource(images=HAGRID_ROOT, annotations=ANNOTATIONS),
    ultralytics=(UltralyticsHandSource(root=ULTRALYTICS_ROOT) if USE_ULTRALYTICS else None),
)
print(json.dumps(stats["totals"], indent=2))

In [ ]:
import json

print((OUT / "data.yaml").read_text())
stats = json.loads((OUT / "stats.json").read_text())
print(json.dumps(stats["splits"], indent=2))
assert stats["imgsz_hw"] == [256, 320]
assert stats["kpt_shape"] == [21, 3]
assert stats["look"]["clahe_clip"] == 3.0 and stats["look"]["clahe_tiles"] == [8, 8]
assert stats["splits"]["train"]["images_written"] > 0
if "license_warning" in stats:
    # The Ultralytics hand-keypoints source is CC BY-NC-SA 4.0: the built
    # frames inherit a non-commercial restriction.  Set USE_ULTRALYTICS = False
    # and rebuild before publishing the dataset for anything commercial.
    print("LICENSE WARNING:", stats["license_warning"])

In [ ]:
from pathlib import Path

from cpsa_ml.models import config_path
from cpsa_ml.train import TrainConfig, train

VARIANT = "n"
ACTIVATION = "lrelu"  # DPU-friendly LeakyReLU(26/256)
EPOCHS = 60
BATCH = 64
DEVICE = "0,1"  # 2x T4 DDP
IMGSZ = 320  # square letterbox; the exported graph is 256x320
EXTRA = {
    "hsv_h": 0,  # gray frames: no hue or saturation augmentation
    "hsv_s": 0,
    "hsv_v": 0.3,
    "degrees": 180,  # a hand seen from above has no canonical orientation
    "fliplr": 0.5,
    "mosaic": 1.0,
    "scale": 0.5,
}
PROJECT = Path("/kaggle/working/runs")
NAME = "hand-topdown-yolov8n-lrelu"
RESUME = ""  # e.g. "/kaggle/input/cpsa-hand-topdown-model/last.pt"

best = train(
    TrainConfig(
        data=OUT / "data.yaml",
        schema="hand21",
        variant=VARIANT,
        activation=ACTIVATION,
        model_cfg=config_path("yolov8n-hand-lrelu.yaml"),
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        imgsz=IMGSZ,
        project=PROJECT,
        name=NAME,
        resume=Path(RESUME) if RESUME else None,
        seed=0,
        workers=4,
        patience=15,
        amp=True,
        val=True,
        plots=True,
        extra=EXTRA,
    )
)
print("best checkpoint:", best)

In [ ]:
import shutil
from pathlib import Path

from cpsa_ml.onnx_check import export_onnx

MODEL = Path("/kaggle/working/hand-topdown-model")
MODEL.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, MODEL / "best.pt")
onnx_path = export_onnx(best, MODEL / "hand-topdown-256x320.onnx", imgsz=(256, 320))
for name in ("data.yaml", "schema.json", "stats.json"):
    shutil.copy2(OUT / name, MODEL / name)
(MODEL / "train_args.json").write_text(
    (PROJECT / NAME / "args.yaml").read_text(encoding="utf-8"), encoding="utf-8"
)
print("onnx:", onnx_path)
for path in sorted(MODEL.iterdir()):
    print(path.name, path.stat().st_size)

## Next steps

Re-publish `/kaggle/working/hand-topdown` as the dataset
`fabiocabeccia/cpsa-hand-topdown` (only if the CC BY-NC-SA source is acceptable)
and `/kaggle/working/hand-topdown-model` as `fabiocabeccia/cpsa-hand-topdown-model`.
The DPU audit runs on the exported 256x320 graph, and the runtime prefilters the
IR frame with the same CLAHE before it reaches the model.

Booth recordings can extend the same dataset without Kaggle: the INZI frames
captured at the booth are already gray, so they only need the window and the
labels:

```
ml/.venv/bin/python -m cpsa_ml.datasets.handframe --ultralytics-root data/hand-keypoints --out data/hand-topdown
```
